In [27]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file, 'r') as file:
        content = file.read()

    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc.md")

for i, chunk in enumerate(chunks):
    print(f"[{i}] {chunk}\n")

[0] #微軟實習故事：程式碼、西雅圖與未來的形狀

[1] 2026 年的夏天，台北微軟辦公室的玻璃帷幕正反射著耀眼的烈日。陳宇軒（Ethan）站在 19 樓的落地窗前，手裡握著那張剛發下來、印有經典四色微軟標誌的實習生識別證。作為台灣大學資訊工程研究所一年級的學生，他歷經了三輪嚴苛的技術面試與一場關卡重重的行為面試，才終於從數千名申請者中脫穎而出，成為今年「微軟未來生涯體驗計劃（FY26 Intern Program）」的雲端架構部研發實習生。他的心中既充滿了對科技巨擘的憧憬，也夾雜著一絲對於未知挑戰的焦慮。

[2] 帶領宇軒的導師（Mentor）是一位名叫張菡文（Hannah）的高級雲端架構師。菡文在微軟已經服務了八年，眼神裡透著精明與幹練，但笑起來卻十分溫暖。第一天報到時，菡文沒有讓宇軒去看那些枯燥的員工守則，而是直接將他帶到了一間名為「Ada Lovelace」的會議室。菡文在白板上寫下了一個代號——「藍天專案（Project AzureSky）」。這就是宇軒未來兩個月實習的核心任務：利用微軟最新的 Azure OpenAI 服務與分散式向量資料庫，為亞太區的大型金融客戶構建一套具備高併發、低延遲特性的企業級 RAG 知識管理系統。

[3] 「宇軒，這個專案不是實驗室裡的玩具。」菡文看著他，語氣嚴肅而專注。「客戶的原始文件高達數百萬筆，包含複雜的法律條文和財務報表。目前的開源解決方案在處理長文本時，召回率（Recall Rate）只有不到百分之七十，且常常出現幻覺。你的任務是在八週內，將檢索精準度提升至百分之九十以上，並且必須在微軟全球實習生黑客松（Global Intern Hackathon）上進行成果展示。這會是你的終期考核。」

[4] 回到座位後，宇軒打開了微軟為他配備的 Surface Laptop Studio。看著空白的 VS Code 視窗，他深吸了一口氣。他知道，要解決長文本財務報表的檢索問題，傳統的簡單切塊（Chunking）絕對行不通。那些表格數據和跨頁面的關聯資訊，在切塊後會徹底失去上下文。他必須設計出一套創新的雙路檢索機制（Two-Way Retrieval）：結合密集向量檢索（Dense Retrieval）與稀疏關鍵字檢索（BM25），並在最上層加上重排模型（Reranker）。

[5] 實習的第一週，宇軒幾乎泡在微軟

In [28]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()


embedding = embed_chunk("測試文本")
print(len(embedding))
print(embedding)

768
[0.01015530712902546, 0.021729689091444016, 0.028211139142513275, 0.05765268951654434, 0.0070283194072544575, -0.08017019182443619, -0.011021432466804981, -0.05130689591169357, -0.010142137296497822, 0.03034563548862934, -0.007976934313774109, -0.003991974983364344, 0.05656474828720093, -0.035579681396484375, -0.03723400458693504, 0.005211402662098408, -0.002243808237835765, 0.038997750729322433, -0.02246141992509365, 0.01812991313636303, -0.00031300951377488673, 0.009411772713065147, -0.043729089200496674, -0.023011287674307823, 0.03080981783568859, -0.03820257633924484, -0.027890656143426895, 0.02508680708706379, 0.024516835808753967, 0.021302273496985435, 0.015752289444208145, -0.03584206849336624, -0.04484609141945839, 0.04726772755384445, -0.032924313098192215, -0.007873951457440853, 0.016273846849799156, -0.030341945588588715, 0.028517939150333405, -0.00885583832859993, 0.006700324825942516, 0.011326364241540432, -0.07289304584264755, -0.020821448415517807, 0.0037962477654218

In [29]:
# Embed all chunks
embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(embeddings[0])

24
[-0.004671085625886917, 0.001112015568651259, 0.059089187532663345, 0.031218569725751877, -0.022259961813688278, -0.08043735474348068, 0.040451642125844955, 0.005833642557263374, -0.018511153757572174, -0.01160359289497137, -0.011405348777770996, -0.009452586062252522, 0.00795633066445589, -0.025269918143749237, -0.03328798711299896, 0.004979380872100592, -0.0033468245528638363, 0.015165796503424644, -0.0075850049033761024, 0.008783558383584023, -0.004276790656149387, 0.06960504502058029, 0.003766509471461177, 0.03125683590769768, -0.008558038622140884, -0.010400323197245598, -0.01860004849731922, 0.03161892667412758, -0.01742635667324066, 0.05299733951687813, 0.02176067791879177, -0.044759925454854965, -0.0673779770731926, 0.04862948879599571, -4.550558878690936e-05, -0.018822146579623222, 0.02163088321685791, 0.006117433775216341, 0.018090633675456047, -0.052638810127973557, 0.040321383625268936, 0.06246567144989967, -0.04257962107658386, -0.024342486634850502, -0.0124142002314329

In [30]:
# Save embeddings to ChromaDB
import chromadb

chromadb_client = chromadb.EphemeralClient() # Use in-memory ChromaDB for testing; switch to PersistentClient("./chroma.db") for production    
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

# Save embeddings(including IDs, vectors) to ChromaDB
def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        chromadb_collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

save_embeddings(chunks, embeddings)

In [31]:
# Retrieve relevant chunks based on query
def retrieve(query: str, top_k: int) -> List[str]:
    query_embedding = embed_chunk(query) # Embed the query using the same embedding model
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0] # Return the retrieved chunks

query = "故事中林佳妤（Chloe）針對 Azure Cosmos DB 的寫入崩潰問題，具體提出了哪三個架構優化建議？"
retrieved_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

[0] 當簡報結束，台下掌聲雷動。總經理親自站起來向他提問：「Ethan，如果未來這套系統要支持每秒數萬次的全球併發查詢，你認為目前的架構最大的瓶頸會在哪裡？你會如何優化？」宇軒自信地笑了笑，回答道：「目前的瓶頸會主要在於 LLM 的 Token 生成速度與成本。未來我們可以考慮引入快取機制（Semantic Caching），將相似度極高的常見問題解答緩存在 Redis 中，減少對大模型的直接調用；同時可以針對特定領域微調一個輕量級的開源小模型（SLM），部署在 Azure Kubernetes Service 上，以降低整體營運成本。」這個回答展現了他超越一般實習生的架構思維。

[1] 然而，現實的開發過程從來不會是一帆風順的。到了第三週，當宇軒將處理好的 50 萬筆測試數據導入 Azure Cosmos DB for MongoDB 的向量索引時，系統崩潰了。終端機上彈出了一連串血紅色的錯誤訊息：「Request rate is large (ExceededTimeLimit)」。由於大量的並行寫入需求，觸發了資料庫的吞吐量限制（RU/s 限制）。此時距離中期進度審查（Mid-term Review）只剩下不到四十八小時，宇軒的額頭滲出了冷汗。

[2] 除了技術指標達標外，微軟非常重視資料隱私與安全（Responsible AI）。金融客戶的資料絕對不能洩漏到公有雲的公共模型中。因此，宇軒在最後一週的優化中，專門加入了基於 Azure Presidio 的隱私資料去識別化（Data Masking）模組。在所有使用者查詢送入大模型之前，系統會自動將姓名、身分證字號、銀行帳號等敏感資訊進行遮蔽，並在回傳答案時進行還原。這個安全的設計，成為了他畢業答辯簡報中最亮眼的一頁。

[3] 當宇軒修復完最後一個 Bug，將代碼推送到 GitHub 時，外面的天空已經泛起了魚肚白。黑客松的最後一天的 Demo Market 展位上，「Azure Pioneers」的攤位前擠滿了來自各部門的評審與工程師。梓豪用生動流暢的簡報吸引了所有人的目光，而宇軒則現場演示了「FinNavigator」如何在五秒內精準分析出一份兩百頁財報的潛在風險，並給出帶有引用來源的回答。系統表現得無懈可擊，現場響起了熱烈的掌聲。

[4] 實習的最後一天，台北辦公室舉行了實習生畢業成果發表會（F

In [32]:
# Rerank retrieved chunks using CrossEncoder
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs) # Get relevance scores for each chunk

    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, _ in scored_chunks][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

[0] 然而，現實的開發過程從來不會是一帆風順的。到了第三週，當宇軒將處理好的 50 萬筆測試數據導入 Azure Cosmos DB for MongoDB 的向量索引時，系統崩潰了。終端機上彈出了一連串血紅色的錯誤訊息：「Request rate is large (ExceededTimeLimit)」。由於大量的並行寫入需求，觸發了資料庫的吞吐量限制（RU/s 限制）。此時距離中期進度審查（Mid-term Review）只剩下不到四十八小時，宇軒的額頭滲出了冷汗。

[1] 實習的最後一天，台北辦公室舉行了實習生畢業成果發表會（Final Presentation）。宇軒站在大會議室的講臺上，面對著包括台灣微軟總經理、研發協理以及所有導師在內的三 layout 評審團。他用沉穩的口吻，從金融長文本的切塊痛點講起，延伸到分散式向量資料庫的寫入優化，再到雙路檢索與隱私安全的架構設計。他的簡報邏輯嚴密、數據詳實，完美的將技術深度與商業價值結合在一起。

[2] 當簡報結束，台下掌聲雷動。總經理親自站起來向他提問：「Ethan，如果未來這套系統要支持每秒數萬次的全球併發查詢，你認為目前的架構最大的瓶頸會在哪裡？你會如何優化？」宇軒自信地笑了笑，回答道：「目前的瓶頸會主要在於 LLM 的 Token 生成速度與成本。未來我們可以考慮引入快取機制（Semantic Caching），將相似度極高的常見問題解答緩存在 Redis 中，減少對大模型的直接調用；同時可以針對特定領域微調一個輕量級的開源小模型（SLM），部署在 Azure Kubernetes Service 上，以降低整體營運成本。」這個回答展現了他超越一般實習生的架構思維。



In [38]:
from dotenv import load_dotenv
from google import genai
import time
import random
from google.genai.errors import APIError  # 用於精準捕捉 Google API 錯誤

# 載入 .env 檔案中的環境變數
load_dotenv() 

# 初始化 Google Client
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    prompt = f"""你是一個知識助手，根據用戶的問題和下列片段生成準確的的答案

用戶問題: {query}

相關片段:
{"\n\n".join(chunks)}

基於上述內容回答，不要編造。若沒有找到相關資訊，請回答「抱歉，我無法從提供的片段中找到答案。」"""

    print(f"{prompt}\n\n---\n")

    base_delay = 1  # 基礎等待時間（秒）
    max_attempts = 5  # 最多重試 5 次

    for attempt in range(max_attempts):
        try:
            # 呼叫 Gemini 2.5 Flash 模型
            response = google_client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text

        except APIError as e:
            # 檢查是否為 503 UNAVAILABLE 錯誤
            if e.code == 503:
                # 計算指數退避時間，並加上隨機微調（Jitter）避免同時衝撞伺服器
                delay = (base_delay * (2 ** attempt)) + random.uniform(0, 1)
                print(f"⚠️ [503] 伺服器高負載忙碌中。將在 {delay:.2f} 秒後進行第 {attempt + 1} 次重試...")
                time.sleep(delay)
            else:
                # 其他 API 錯誤（如 400 參數錯誤、403 權限錯誤）直接拋出，不進行重試
                raise e
        except Exception as e:
            # 捕捉其他非 API 的突發錯誤
            raise e

    raise Exception(f" 錯誤：已嘗試重試 {max_attempts} 次，Gemini 伺服器依舊忙碌無法回應。")

# 測試呼叫（請確保您的 query 與 reranked_chunks 已在外部定義）
answer = generate(query, reranked_chunks)
print(answer)


你是一個知識助手，根據用戶的問題和下列片段生成準確的的答案

用戶問題: 故事中林佳妤（Chloe）針對 Azure Cosmos DB 的寫入崩潰問題，具體提出了哪三個架構優化建議？

相關片段:
然而，現實的開發過程從來不會是一帆風順的。到了第三週，當宇軒將處理好的 50 萬筆測試數據導入 Azure Cosmos DB for MongoDB 的向量索引時，系統崩潰了。終端機上彈出了一連串血紅色的錯誤訊息：「Request rate is large (ExceededTimeLimit)」。由於大量的並行寫入需求，觸發了資料庫的吞吐量限制（RU/s 限制）。此時距離中期進度審查（Mid-term Review）只剩下不到四十八小時，宇軒的額頭滲出了冷汗。

實習的最後一天，台北辦公室舉行了實習生畢業成果發表會（Final Presentation）。宇軒站在大會議室的講臺上，面對著包括台灣微軟總經理、研發協理以及所有導師在內的三 layout 評審團。他用沉穩的口吻，從金融長文本的切塊痛點講起，延伸到分散式向量資料庫的寫入優化，再到雙路檢索與隱私安全的架構設計。他的簡報邏輯嚴密、數據詳實，完美的將技術深度與商業價值結合在一起。

當簡報結束，台下掌聲雷動。總經理親自站起來向他提問：「Ethan，如果未來這套系統要支持每秒數萬次的全球併發查詢，你認為目前的架構最大的瓶頸會在哪裡？你會如何優化？」宇軒自信地笑了笑，回答道：「目前的瓶頸會主要在於 LLM 的 Token 生成速度與成本。未來我們可以考慮引入快取機制（Semantic Caching），將相似度極高的常見問題解答緩存在 Redis 中，減少對大模型的直接調用；同時可以針對特定領域微調一個輕量級的開源小模型（SLM），部署在 Azure Kubernetes Service 上，以降低整體營運成本。」這個回答展現了他超越一般實習生的架構思維。

基於上述內容回答，不要編造。若沒有找到相關資訊，請回答「抱歉，我無法從提供的片段中找到答案。」

---

⚠️ [503] 伺服器高負載忙碌中。將在 1.56 秒後進行第 1 次重試...
抱歉，我無法從提供的片段中找到答案。片段中並未提及林佳妤（Chloe）針對 Azure Cosmos DB 寫入崩潰問題提出的架構優化建議。片段中描述遇到寫入崩潰問題的是